In [0]:
"""forEach runner: Reads a function template from the repo, substitutes
target-specific variables, and executes the DDL via spark.sql().

Parameters (from forEach iteration + base_parameters):
  function_file: filename in uc_functions/ dir (e.g. "create_plan.sql")
  catalog: target UC catalog
  schema: target UC schema
  connection_name: metastore-level HTTP connection name
  workers_group_name: group receiving EXECUTE grant

Path derivation: auto-discovers uc_functions/ as a sibling directory
to this notebook's location (works with source-linked deployments).
"""
import os

# -- Parameters --
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
connection_name = dbutils.widgets.get("connection_name")
workers_group_name = dbutils.widgets.get("workers_group_name")
function_file = dbutils.widgets.get("function_file")

# -- Derive uc_functions/ path from this notebook's location --
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
parent_dir = os.path.dirname(nb_path)
uc_functions_dir = os.path.join(parent_dir, "uc_functions")
fs_path = f"/Workspace{uc_functions_dir}"

print(f"Function file: {function_file}")
print(f"UC functions dir: {fs_path}")
print(f"Target: {catalog}.{schema}")

# -- Read template file --
template_path = os.path.join(fs_path, function_file)
with open(template_path, "r") as f:
    template = f.read()

print(f"\nTemplate loaded ({len(template)} chars)")

# -- Substitute tokens --
ddl = template.replace("{{catalog}}", catalog)
ddl = ddl.replace("{{schema}}", schema)
ddl = ddl.replace("{{connection_name}}", connection_name)

# Strip comment-only lines (-- ...) from the top for cleaner execution
lines = ddl.strip().split("\n")
while lines and lines[0].strip().startswith("--"):
    lines.pop(0)
ddl = "\n".join(lines).strip()

# -- Derive function name from filename --
func_name = function_file.replace(".sql", "").replace("-", "_")
full_name = f"{catalog}.{schema}.{func_name}"

print(f"Registering: {full_name}")

# -- Drop existing (idempotent) --
spark.sql(f"DROP FUNCTION IF EXISTS {full_name}")

# -- Create the function --
spark.sql(ddl)
print(f"  Created.")

# -- Grant EXECUTE to workers group --
spark.sql(f"GRANT EXECUTE ON FUNCTION {full_name} TO `{workers_group_name}`")
print(f"  Granted EXECUTE to {workers_group_name}")

print(f"\n✅ Registered: {full_name}")